In [ ]:
import sys, subprocess, json, time
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=5.8,<6', 'accelerate>=1.10', 'bitsandbytes>=0.48'])
import torch, transformers
from transformers import AutoTokenizer, Qwen3_5ForConditionalGeneration, BitsAndBytesConfig
MODEL_ID = 'Qwen/Qwen3.8-27B'
MODEL_REVISION = '1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0'
print('VERSIONS', torch.__version__, transformers.__version__, flush=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, device_map={'': 0}, dtype=torch.bfloat16,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True),
    attn_implementation='sdpa')
model.eval()
messages=[{'role':'user','content':'Reply with exactly: MODEL_READY'}]
prompt=tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs=tokenizer(prompt, return_tensors='pt').to('cuda')
with torch.inference_mode():
    out=model.generate(**inputs, max_new_tokens=32, do_sample=False)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:],skip_special_tokens=True))
print('GPU_ALLOCATED_GIB',round(torch.cuda.memory_allocated()/2**30,2))


VERSIONS 2.11.0+cu130 5.20.2


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


In [3]:
import json, subprocess, pathlib
status = {'model_loaded': 'model' in globals(), 'gpu': subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,memory.used','--format=csv'],text=True)}
if 'model' in globals(): status['model_type']=type(model).__name__
pathlib.Path('/content/runtime_status.json').write_text(json.dumps(status))
print(json.dumps(status), flush=True)


{"model_loaded": true, "gpu": "name, memory.total [MiB], memory.used [MiB]\nNVIDIA A100-SXM4-40GB, 40960 MiB, 18214 MiB\n", "model_type": "Qwen3_5ForConditionalGeneration"}


In [4]:
import zipfile, os, sys
from pathlib import Path
with zipfile.ZipFile('/content/commitment-pilot-runtime.zip') as z:
    z.extractall('/content')
ROOT=Path('/content/commitment-pilot')
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
SELECT_IDS=['mb_00','syn_01']
exec((ROOT/'colab_run.py').read_text())


START mb_00
COMPLETE mb_00 meetingbank dev
[
  {
    "domain": "meetingbank",
    "split": "dev",
    "method": "generated_summary",
    "correct": 3,
    "n": 3,
    "accuracy": 1.0,
    "parse_failure_questions": 0
  },
  {
    "domain": "meetingbank",
    "split": "dev",
    "method": "reference_summary",
    "correct": 3,
    "n": 3,
    "accuracy": 1.0,
    "parse_failure_questions": 0
  },
  {
    "domain": "meetingbank",
    "split": "dev",
    "method": "tracker",
    "correct": 3,
    "n": 3,
    "accuracy": 1.0,
    "parse_failure_questions": 0
  },
  {
    "domain": "meetingbank",
    "split": "dev",
    "method": "transcript",
    "correct": 3,
    "n": 3,
    "accuracy": 1.0,
    "parse_failure_questions": 0
  }
]
START syn_01
COMPLETE syn_01 synthetic dev
[
  {
    "domain": "meetingbank",
    "split": "dev",
    "method": "generated_summary",
    "correct": 3,
    "n": 3,
    "accuracy": 1.0,
    "parse_failure_questions": 0
  },
  {
    "domain": "meetingbank",
    "spl

In [5]:
# Final pilot configuration: synthetic turns processed separately; question-only control.
import zipfile, importlib, shutil
assert len(all_results)==2, 'Finish and inspect the development smoke test first.'
assert not any(c['errors'] for r in all_results for c in r['conditions']), 'Inspect malformed development answers first.'
with zipfile.ZipFile('/content/commitment-pilot-runtime-v02.zip') as z:
    z.extractall('/content')
import pilot
importlib.reload(pilot)
print('RUNNER_VERSION', pilot.VERSION, flush=True)
SELECT_IDS=None
try:
    exec((ROOT/'colab_run.py').read_text())
finally:
    archive=shutil.make_archive('/content/commitment-pilot-results','zip',str(ROOT))
    print('RESULT_ARCHIVE',archive,flush=True)


RUNNER_VERSION 0.2.0
START mb_00
COMPLETE mb_00 meetingbank dev
[
  {
    "domain": "meetingbank",
    "split": "dev",
    "method": "generated_summary",
    "correct": 3,
    "n": 3,
    "accuracy": 1.0,
    "parse_failure_questions": 0
  },
  {
    "domain": "meetingbank",
    "split": "dev",
    "method": "question_only",
    "correct": 3,
    "n": 3,
    "accuracy": 1.0,
    "parse_failure_questions": 0
  },
  {
    "domain": "meetingbank",
    "split": "dev",
    "method": "reference_summary",
    "correct": 3,
    "n": 3,
    "accuracy": 1.0,
    "parse_failure_questions": 0
  },
  {
    "domain": "meetingbank",
    "split": "dev",
    "method": "tracker",
    "correct": 3,
    "n": 3,
    "accuracy": 1.0,
    "parse_failure_questions": 0
  },
  {
    "domain": "meetingbank",
    "split": "dev",
    "method": "transcript",
    "correct": 3,
    "n": 3,
    "accuracy": 1.0,
    "parse_failure_questions": 0
  }
]
START mb_06
COMPLETE mb_06 meetingbank dev
[
  {
    "domain": "meeti

In [6]:
import json
scored = json.loads((ROOT/'results/scores_draft.json').read_text())
print('COMPLETED EXAMPLES:', len(all_results), '/ 22')
print('AI-DRAFT LABELS: independent human review is pending.\n')
for domain in ['meetingbank', 'synthetic']:
    print(domain.upper(), '- PILOT SPLIT ONLY')
    for method in ['question_only','transcript','reference_summary','generated_summary','tracker']:
        row = next((r for r in scored['summary'] if r['domain']==domain and r['split']=='pilot' and r['method']==method), None)
        if row:
            print(f"  {method:20s} {row['correct']:2d}/{row['n']:2d}  {100*row['accuracy']:5.1f}%")
    print()
print('Checkpoint ZIP:', '/content/commitment-pilot-results.zip')
print('QA accuracy is not record extraction F1 or evidence entailment.')


COMPLETED EXAMPLES: 22 / 22
AI-DRAFT LABELS: independent human review is pending.

MEETINGBANK - PILOT SPLIT ONLY
  question_only        17/24   70.8%
  transcript           24/24  100.0%
  reference_summary    15/24   62.5%
  generated_summary    22/24   91.7%
  tracker              21/24   87.5%

SYNTHETIC - PILOT SPLIT ONLY
  question_only        10/24   41.7%
  transcript           24/24  100.0%
  generated_summary    24/24  100.0%
  tracker              24/24  100.0%

Checkpoint ZIP: /content/commitment-pilot-results.zip
QA accuracy is not record extraction F1 or evidence entailment.
